# Notebook 28: Attention-Conditioned Diffusion U-Net

---

## What This Notebook Covers

This notebook assembles everything from the diffusion track into a single, trainable generator: a **U-Net that predicts the noise in a diffusion process**, conditioned on the *time step* (how noisy the image is) and augmented with **self-attention** at its lower-resolution stages. We then sample from it with **DDIM**, score it with **FID/KID**, and finally build a **class-conditional** variant that generates a *chosen* Fashion-MNIST category on demand. We will learn:

1. **Sinusoidal timestep embeddings** — how a scalar noise level `t ∈ [0,1]` becomes a rich vector the network can read (`timestep_embedding`)
2. **FiLM-style time conditioning** — how that embedding modulates every residual block via a learned per-channel `scale` and `shift` (`EmbResBlock`)
3. **Self-attention inside the U-Net** — reusing the from-scratch attention of notebook 27 as `SelfAttention2D`, placed only where it pays off (low resolution)
4. **The `saved()` forward-hook trick** — a tiny monkeypatch that captures intermediate activations for the U-Net's skip connections
5. **The full `EmbUNetModel`** — down blocks, a mid block, up blocks, and the skip-connection channel bookkeeping that makes them fit together
6. **DDIM sampling** — the `ddim_step` update rule, `x₀` prediction, and the `eta` knob that interpolates between stochastic DDPM and deterministic DDIM
7. **FID / KID evaluation** — putting a number on sample quality
8. **Class conditioning** — adding a learned class embedding to the time embedding so the model generates a *specified* label (`CondUNetModel`, `cond_sample`)

This is an `#|export` notebook: the cells marked `#| export` compile into `miniai.diffusion`, so what we build here becomes a reusable library module.

---

## Why Attention + Conditioning in Diffusion?

By notebook 26 we had a diffusion U-Net that could turn noise into Fashion-MNIST images, but it had two limitations this notebook removes.

**First: purely local reasoning.** A plain conv U-Net sees the world through small kernels. At the coarsest (most downsampled) stage of the U-Net, where the whole 32×32 image has been squeezed into an 4×4 grid of rich feature vectors, we want a *global* operation that asks "does this whole image hang together?" That is exactly what self-attention provides — and notebook 27 built it from scratch. Here we drop it in, but **only at low resolution**, because attention's `O(S²)` cost over `S = H·W` positions is affordable when `S` is small and ruinous when it is large. This "attention where the sequence is short" placement is a design decision worth internalizing.

**Second: no control.** An unconditional generator produces *some* plausible image; it cannot be told *which*. The conditional model at the end adds a learned **class embedding** to the time embedding, so a single trained network can be asked for a "Sandal" or an "Ankle boot." This is the seed of everything from class-conditional ImageNet models to text-conditioned Stable Diffusion — the conditioning signal changes, the mechanism (inject an embedding into the residual blocks) does not.

**Climate / EO bridges (real ones).**
- **Conditioning ↔ constrained weather/field generation.** A class label steering a generator is the toy version of conditioning a generative weather emulator on a forcing (SST anomaly, greenhouse-gas scenario, a coarse GCM field to downscale). The plumbing — embed the conditioning variable, add it into the network's residual stream — is identical.
- **Low-res attention ↔ global field coherence / teleconnections.** Placing attention at the U-Net bottleneck is the diffusion analog of letting a spatial field's far-apart regions communicate directly (the teleconnection-matrix idea from notebook 27), which local convolutions cannot express.
- **`x₀` prediction in DDIM ↔ analysis increment.** The sampler's "estimate the clean image, then take a controlled step back toward it" loop rhymes with iterative NWP data-assimilation updates.

---

## Prerequisites

You should be comfortable with:

- **Self-attention from scratch** (notebook 27) — `SelfAttention`, the `rearrange('n s (h d) -> (n h) s d')` multi-head trick, `√(ni/nheads)` scaling. This notebook *reuses* that code.
- **The diffusion U-Net** (notebook 26) — down/up blocks, skip connections, time embedding.
- **DDPM and DDIM** (notebooks 15–20) — the forward noising process `xₜ = √ᾱₜ·x₀ + √(1−ᾱₜ)·ε`, predicting ε, and DDIM's faster deterministic sampling.
- **FID / KID** (notebook 18) — Fréchet/Kernel Inception Distance for scoring generated images.
- **`miniai` `Learner` + callbacks** (notebooks 09–12) — `DeviceCB`, `MetricsCB`, `MixedPrecision`, `OneCycleLR`.
- **Residual blocks and pre-activation conv** (notebook 13).

---


# Part 1: Setup, the Noise Schedule, and Data

We start with imports, a fixed GPU choice, reproducibility settings, and the diffusion **noising** machinery — a cosine `ᾱ` (alpha-bar) schedule and a `noisify` function that corrupts a clean image to a random noise level. If you did notebooks 19–22 this is familiar; the only thing to re-anchor is that `noisify` returns the *inputs* `(xₜ, t)` and the *target* `ε` (the noise), because the network's job is to predict the noise.

---


In [ ]:
#| default_exp diffusion

**What does the code above do?**

`#| default_exp diffusion` is an **nbdev** directive: every cell below marked `#| export` will be compiled into `miniai/diffusion.py`. So this notebook is not just a lesson — it is the *source* of the `miniai.diffusion` module. (Do not remove or add `#| export` markers casually; they decide what lands in the library.)


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='1'

**What does the code above do?**

Pins training to GPU index 1 *before* any CUDA context is created (which is why it is set via an environment variable at the very top). On a single-GPU machine you would set this to `'0'` or delete the cell.


In [ ]:
#| export
from miniai.imports import *

from einops import rearrange
from fastprogress import progress_bar

**What does the code above do?**

`from miniai.imports import *` pulls in the whole accumulated toolkit we have been building (torch, `fastcore` as `fc`, the `Learner`, callbacks, `show_images`, `set_seed`, etc.). `rearrange` (einops) is for the multi-head attention reshape from notebook 27; `progress_bar` drives the sampling loop's progress display. This cell is exported, so `miniai.diffusion` inherits these imports.


In [ ]:
torch.set_printoptions(precision=4, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'
mpl.rcParams['figure.dpi'] = 70

import logging
logging.disable(logging.WARNING)

set_seed(42)
if fc.defaults.cpus>8: fc.defaults.cpus=8

**What does the code above do?**

Boilerplate for legible output and reproducibility: readable tensor printing, a reversed-gray colormap (so Fashion-MNIST looks natural), a modest figure DPI, silenced warnings, a global seed, and a cap of 8 CPU workers for data loading. None of it affects the model — it just makes the notebook pleasant and deterministic.


In [ ]:
xl,yl = 'image','label'
name = "fashion_mnist"
bs = 512
dsd = load_dataset(name)

**What does the code above do?**

Loads Fashion-MNIST via HuggingFace `datasets`. `xl='image'` and `yl='label'` name the feature keys we will index with throughout. Batch size 512. `dsd` is a `DatasetDict` with `train`/`test` splits.


In [ ]:
#| export
def abar(t): return (t*math.pi/2).cos()**2
def inv_abar(x): return x.sqrt().acos()*2/math.pi

def noisify(x0):
    device = x0.device
    n = len(x0)
    t = torch.rand(n,).to(x0).clamp(0,0.999)
    ε = torch.randn(x0.shape, device=device)
    abar_t = abar(t).reshape(-1, 1, 1, 1).to(device)
    xt = abar_t.sqrt()*x0 + (1-abar_t).sqrt()*ε
    return (xt, t.to(device)), ε

def collate_ddpm(b): return noisify(default_collate(b)[xl])
def dl_ddpm(ds): return DataLoader(ds, batch_size=bs, collate_fn=collate_ddpm, num_workers=4)

**What does the code above do?**

This is the **forward diffusion process** in code — the continuous-time cosine schedule from notebooks 21–22.

| Piece | Meaning |
|-------|---------|
| `abar(t) = cos(t·π/2)²` | The signal-retention schedule `ᾱ(t)`. At `t=0`, `ᾱ=1` (all signal). At `t=1`, `ᾱ=0` (all noise). Continuous in `t`, no 1000-step table needed. |
| `inv_abar(x)` | The inverse — given an `ᾱ` value, recover the `t` that produced it (used elsewhere for schedule manipulation). |
| `noisify(x0)` | Pick a random noise level `t∈[0,1)` per image, draw noise `ε`, and form `xₜ = √ᾱₜ·x₀ + √(1−ᾱₜ)·ε`. Returns `((xₜ, t), ε)`. |

The key structural point: **the model's input is `(xₜ, t)` and its target is `ε`.** We train the U-Net to look at a noisy image *and be told how noisy it is* (`t`), and to predict the noise that was added. `collate_ddpm` applies `noisify` at batch-collation time (so noising happens on the fly, differently every epoch), and `dl_ddpm` wraps it in a `DataLoader`.

> Note the literal Greek `ε` as a variable name — that is valid Python 3 and Jeremy uses it deliberately to keep the code matching the math `xₜ = √ᾱₜ x₀ + √(1−ᾱₜ)ε`.


In [ ]:
@inplace
def transformi(b): b[xl] = [F.pad(TF.to_tensor(o), (2,2,2,2))-0.5 for o in b[xl]]

tds = dsd.with_transform(transformi)
dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))

dl = dls.train
(xt,t),eps = b = next(iter(dl))

**What does the code above do?**

Sets up the actual data pipeline:

- `transformi` converts each PIL image to a tensor, **pads 28×28 → 32×32** (`F.pad(..., (2,2,2,2))`, so the spatial size is a clean power-of-two-friendly 32 for the U-Net's downsampling), and centers pixels to `[-0.5, 0.5]`. `@inplace` mutates the batch dict in place (a `miniai` helper).
- `with_transform` applies it lazily; `DataLoaders` wraps train/test with the noisifying collate.
- The last line grabs one batch so we can inspect shapes: `xt` is the noised images, `t` the per-image noise levels, `eps` the target noise.

After this, `xt.shape` is `(512, 1, 32, 32)` and `t.shape` is `(512,)`.


---

# Part 2: Telling the Network the Noise Level

The network sees a noisy image `xₜ` and must predict the noise. But the *right* answer depends heavily on **how noisy** the image is — at `t≈0` it should barely change the image, at `t≈1` it should predict almost the entire image as noise. So the scalar `t` must be fed in, and fed in *well*. A single float appended somewhere is a weak signal; instead we expand `t` into a high-dimensional **sinusoidal embedding**, exactly as Transformers encode token positions.

---


In [ ]:
#| export
def timestep_embedding(tsteps, emb_dim, max_period= 10000):
    exponent = -math.log(max_period) * torch.linspace(0, 1, emb_dim//2, device=tsteps.device)
    emb = tsteps[:,None].float() * exponent.exp()[None,:]
    emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
    return F.pad(emb, (0,1,0,0)) if emb_dim%2==1 else emb

**What does the code above do?**

Turns each scalar timestep into an `emb_dim`-dimensional vector of sines and cosines at geometrically-spaced frequencies. The deep dive below derives every line — this is a genuinely fundamental construction that reappears in every diffusion and transformer model.


## Deep Dive: Sinusoidal Timestep Embeddings

Why not just feed the raw number `t` into the network? Because a single scalar is a *terrible* input for a neural net to condition on: a `Linear` layer can only apply one linear function of it, and the net has no easy way to build sharp, localized responses to specific noise levels. The sinusoidal embedding solves this by expanding `t` into many features that vary at many different rates — some change fast as `t` moves, some slow — giving the network a basis rich enough to represent any smooth function of `t`.

This is the **same trick** as Transformer positional encodings; here the "position" is the diffusion timestep.

### The construction, line by line

The target formula (for even `emb_dim = D`, with `k = D/2` frequency bands) is:

$$\text{emb}(t) = \big[\sin(t\omega_0), \dots, \sin(t\omega_{k-1}),\ \cos(t\omega_0), \dots, \cos(t\omega_{k-1})\big]$$

with frequencies geometrically spaced from 1 down to `1/max_period`:

$$\omega_i = \exp\!\left(-\frac{\log(\text{max_period})\, \cdot\, i}{k-1}\right) = \text{max_period}^{-i/(k-1)}.$$

---

### Line 1 — build the exponents

```python
exponent = -math.log(max_period) * torch.linspace(0, 1, emb_dim//2, device=tsteps.device)
```

**What it does:** `torch.linspace(0, 1, k)` makes `k = D/2` evenly-spaced values from 0 to 1. Multiplying by `-log(max_period)` stretches them to run from `0` down to `-log(max_period)`.

**Why this design:** these are the *log-frequencies*. Exponentiating them next (`exponent.exp()`) yields frequencies `ω_i` that decay **geometrically** from `1` (when the exponent is 0) to `1/max_period` (when the exponent is `-log(max_period)`). Geometric (not linear) spacing means we cover a huge dynamic range of rates with few bands — the standard choice, `max_period=10000`, spans four orders of magnitude.

---

### Line 2 — outer product of times and frequencies

```python
emb = tsteps[:,None].float() * exponent.exp()[None,:]
```

**What it does:** `tsteps[:,None]` is shape `(n, 1)`; `exponent.exp()[None,:]` is `(1, k)`. Broadcasting multiplies them into an `(n, k)` matrix whose entry `[j, i]` is `t_j · ω_i` — every timestep against every frequency.

**Why this design:** `t·ω` is the *phase* fed into sin/cos. One matrix multiply (well, a broadcast product) gives all phases for the whole batch at once.

---

### Line 3 — sin and cos halves

```python
emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
```

**What it does:** applies `sin` and `cos` to the phase matrix and concatenates along the feature axis, giving `(n, 2k) = (n, D)`.

**Why both sin and cos:** together they encode the phase *unambiguously* (like `(x, y)` on the unit circle vs. just an angle's sine, which is two-to-one). The pair also makes it linearly easy for the network to represent *shifts* in `t`, since a phase shift is a linear mix of sin and cos — a property the downstream `Linear` layers exploit.

---

### Line 4 — odd-dimension guard

```python
return F.pad(emb, (0,1,0,0)) if emb_dim%2==1 else emb
```

**What it does:** if `emb_dim` is odd, `emb_dim//2` rounded down leaves the output one column short (`2k = emb_dim - 1`); pad a single zero column on the right to hit exactly `emb_dim`.

**Why this design:** purely bookkeeping so the function honors its `emb_dim` contract for any dimension, even or odd.

---

### The payoff

`timestep_embedding(t, n_temb)` converts the batch of scalar noise levels into a `(n, n_temb)` matrix of smooth, multi-frequency features. That matrix is then run through a small MLP (`emb_mlp`, seen later) and injected into *every* residual block. The network can now read "how noisy am I?" as easily as it reads the image itself.


---

# Part 3: Small Building Blocks

Three tiny factory functions assemble the repeated pieces of the network: a **pre-activation conv**, an **upsampling** block, and a **pre-activation linear**. "Pre-activation" means *norm → activation → weight*, the ordering popularized by ResNet-v2 and standard in modern U-Nets: it keeps a clean identity path for the residual add and tends to train more stably.

---


In [ ]:
#| export
def pre_conv(ni, nf, ks=3, stride=1, act=nn.SiLU, norm=None, bias=True):
    layers = nn.Sequential()
    if norm: layers.append(norm(ni))
    if act : layers.append(act())
    layers.append(nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias))
    return layers

**What does the code above do?**

Builds a `norm → act → conv` stack (each optional except the conv). Defaults: 3×3 kernel, stride 1, `SiLU` activation (a.k.a. swish, `x·sigmoid(x)` — smooth and standard in diffusion nets), `padding=ks//2` to preserve spatial size. Because the norm and activation come *before* the conv, stacking two of these inside a residual block leaves the skip connection (`+inp`) acting on raw, un-normalized features — the pre-activation pattern.


In [ ]:
#| export
def upsample(nf): return nn.Sequential(nn.Upsample(scale_factor=2.), nn.Conv2d(nf, nf, 3, padding=1))

**What does the code above do?**

Doubles spatial resolution: nearest-neighbor `Upsample` (×2) followed by a 3×3 conv that smooths the upsampled feature map. This "resize-then-conv" upsampling avoids the checkerboard artifacts that transposed convolutions are prone to, and is the standard up-path operation in the U-Net's decoder.


In [ ]:
#| export
def lin(ni, nf, act=nn.SiLU, norm=None, bias=True):
    layers = nn.Sequential()
    if norm: layers.append(norm(ni))
    if act : layers.append(act())
    layers.append(nn.Linear(ni, nf, bias=bias))
    return layers

**What does the code above do?**

The linear-layer analog of `pre_conv`: optional `norm → act → Linear`. Used to build the timestep-embedding MLP (`emb_mlp`) that post-processes the sinusoidal embedding before it is injected into the blocks.


---

# Part 4: Self-Attention for the U-Net

Notebook 27 built self-attention from scratch. Here we adapt it for use *inside* the U-Net. Jeremy shows two versions: a first attempt built on `nn.MultiheadAttention` that "gives poor results," and the from-scratch version (the one that gets exported), wrapped for 2D feature maps as `SelfAttention2D`.

---


In [ ]:
# This version is giving poor results - use the cell below instead
class SelfAttention(nn.Module):
    def __init__(self, ni, attn_chans):
        super().__init__()
        self.attn = nn.MultiheadAttention(ni, ni//attn_chans, batch_first=True)
        self.norm = nn.BatchNorm2d(ni)

    def forward(self, x):
        n,c,h,w = x.shape
        x = self.norm(x).view(n, c, -1).transpose(1, 2)
        x = self.attn(x, x, x, need_weights=False)[0]
        return x.transpose(1,2).reshape(n,c,h,w)

**What does the code above do? (and why it's kept)**

This is a *rejected* attempt, preserved by Jeremy with the comment "giving poor results — use the cell below instead." It uses PyTorch's built-in `nn.MultiheadAttention` with `BatchNorm2d` as the pre-norm, and — importantly — **has no residual connection** (it returns the attention output directly, not `x + attn(x)`).

It is not exported (no `#| export`). The likely culprits for the poor results are the missing residual (so the block cannot easily learn to be a no-op, destabilizing training) and the interaction of `BatchNorm` with the attention statistics. Keeping it here is instructive: it shows that "just call the library layer" is not automatically better than the from-scratch block, and it motivates the next cell.


In [ ]:
#| export
class SelfAttention(nn.Module):
    def __init__(self, ni, attn_chans, transpose=True):
        super().__init__()
        self.nheads = ni//attn_chans
        self.scale = math.sqrt(ni/self.nheads)
        self.norm = nn.LayerNorm(ni)
        self.qkv = nn.Linear(ni, ni*3)
        self.proj = nn.Linear(ni, ni)
        self.t = transpose

    def forward(self, x):
        n,c,s = x.shape
        if self.t: x = x.transpose(1, 2)
        x = self.norm(x)
        x = self.qkv(x)
        x = rearrange(x, 'n s (h d) -> (n h) s d', h=self.nheads)
        q,k,v = torch.chunk(x, 3, dim=-1)
        s = (q@k.transpose(1,2))/self.scale
        x = s.softmax(dim=-1)@v
        x = rearrange(x, '(n h) s d -> n s (h d)', h=self.nheads)
        x = self.proj(x)
        if self.t: x = x.transpose(1, 2)
        return x

**What does the code above do?**

This is the exported, from-scratch multi-head self-attention — the direct descendant of notebook 27's `SelfAttentionMultiHead`, with three practical changes for U-Net use:

| Change vs. nb 27 | Detail | Why |
|------------------|--------|-----|
| **Heads from channels** | `nheads = ni // attn_chans` | You specify *channels-per-head* (`attn_chans`) rather than a head count; the module derives the number of heads. |
| **LayerNorm** | `nn.LayerNorm(ni)` | Replaces BatchNorm/GroupNorm as the pre-norm — batch-independent and the transformer-standard choice. |
| **`transpose` flag** | `self.t` | Lets the block accept either `(n, s, c)` or `(n, c, s)` layouts by transposing on entry/exit. |

The core is unchanged from notebook 27: fused `qkv` projection → `rearrange('n s (h d) -> (n h) s d')` to fold heads into the batch → `chunk` into Q/K/V → scaled scores `(q@kᵀ)/√(ni/nheads)` → softmax → mix → `rearrange` back → output projection.

One subtlety: this version returns `proj(attn(x))` **without** an internal residual — the residual is added by the *caller* (`EmbResBlock` does `x = x + self.attn(x)`), which is a cleaner separation than baking the residual inside.

> Note the variable `s` is reused: first as the sequence-length dim from `n,c,s = x.shape`, then rebound to the score matrix `s = (q@k...)`. Harmless here since the shape unpack isn't needed after, but worth noticing when reading.


In [ ]:
#| export
class SelfAttention2D(SelfAttention):
    def forward(self, x):
        n,c,h,w = x.shape
        return super().forward(x.view(n, c, -1)).reshape(n,c,h,w)

**What does the code above do?**

A thin adapter so the sequence-based `SelfAttention` can be dropped into a 2D conv network. It flattens the `(n, c, h, w)` feature map to `(n, c, h·w)` — the exact **flatten-to-sequence** move from notebook 27 — calls the parent `forward` (which, with `transpose=True`, handles the `(n,c,s)↔(n,s,c)` swap internally), and reshapes the result back to `(n, c, h, w)`. This is the object that actually gets inserted into the residual blocks.


---

# Part 5: The Conditioned Residual Block

`EmbResBlock` is the workhorse repeated throughout the U-Net. It is a residual block that (a) processes features with two pre-activation convs, (b) is **modulated by the timestep embedding** via a learned per-channel scale and shift, and (c) optionally applies self-attention. The conditioning mechanism is the important part and gets a deep dive.

---


In [ ]:
#| export
class EmbResBlock(nn.Module):
    def __init__(self, n_emb, ni, nf=None, ks=3, act=nn.SiLU, norm=nn.BatchNorm2d, attn_chans=0):
        super().__init__()
        if nf is None: nf = ni
        self.emb_proj = nn.Linear(n_emb, nf*2)
        self.conv1 = pre_conv(ni, nf, ks, act=act, norm=norm)
        self.conv2 = pre_conv(nf, nf, ks, act=act, norm=norm)
        self.idconv = fc.noop if ni==nf else nn.Conv2d(ni, nf, 1)
        self.attn = False
        if attn_chans: self.attn = SelfAttention2D(nf, attn_chans)

    def forward(self, x, t):
        inp = x
        x = self.conv1(x)
        emb = self.emb_proj(F.silu(t))[:, :, None, None]
        scale,shift = torch.chunk(emb, 2, dim=1)
        x = x*(1+scale) + shift
        x = self.conv2(x)
        x = x + self.idconv(inp)
        if self.attn: x = x + self.attn(x)
        return x

**What does the code above do?**

A time-conditioned residual block. The constructor wires up:

- `emb_proj`: a `Linear(n_emb, nf*2)` that turns the timestep embedding into **two** per-channel vectors (`scale` and `shift`).
- `conv1`, `conv2`: two pre-activation convs.
- `idconv`: an identity for the skip if channels match (`fc.noop`), else a 1×1 conv to match `ni→nf` so the residual add is shape-compatible.
- `attn`: an optional `SelfAttention2D` (only if `attn_chans>0`).

The `forward` does: conv1 → **apply time modulation `x*(1+scale)+shift`** → conv2 → residual add → optional attention residual. The modulation step is the deep dive below.


## Deep Dive: FiLM — Conditioning by Scale and Shift

How does a *single* network handle images at *every* noise level? It cannot use fixed weights, because the correct behavior at `t≈0` (barely denoise) and `t≈1` (predict everything) are different. The answer is **feature-wise linear modulation (FiLM)**: let the conditioning signal (here, the time embedding) produce a per-channel **scale** and **shift** that rescale the feature maps. The convolution weights stay fixed; the conditioning *reshapes the activations* those weights act on.

### The three lines

```python
emb = self.emb_proj(F.silu(t))[:, :, None, None]   # (n, 2·nf, 1, 1)
scale, shift = torch.chunk(emb, 2, dim=1)          # each (n, nf, 1, 1)
x = x*(1 + scale) + shift                          # broadcast over H, W
```

**Line 1 — project the embedding to `2·nf` numbers.** `t` here is the *processed* timestep embedding (an `(n, n_emb)` matrix). `F.silu(t)` applies the activation, then `emb_proj` (a `Linear(n_emb, nf*2)`) maps it to `(n, 2·nf)`. The `[:, :, None, None]` adds two trailing singleton axes → `(n, 2·nf, 1, 1)`, so it can broadcast across the spatial dimensions of the feature map.

**Line 2 — split into scale and shift.** `torch.chunk(emb, 2, dim=1)` cuts the `2·nf` channels into two `(n, nf, 1, 1)` tensors: one scale value and one shift value **per channel, per image**.

**Line 3 — modulate.** `x*(1 + scale) + shift`. Two design choices matter:

- **`1 + scale`, not `scale`.** At initialization `emb_proj` outputs ≈0, so `scale≈0` and the factor is `≈1` — the block starts as an *identity* modulation and does not disrupt the feature scale. The network then *learns* how much to deviate from 1. (Compare to multiplying by a raw `scale≈0`, which would zero the features at the start and stall learning.)
- **Per-channel, spatially uniform.** `scale`/`shift` are `(n, nf, 1, 1)` — one number per channel, broadcast across all `H×W` positions. So the time embedding says "turn channel 7 up, channel 12 down, everywhere in this image," a global, content-independent modulation. That is exactly the right granularity for a *global* condition like "how noisy is this image."

### Why FiLM instead of concatenating `t` as an extra channel?

Concatenation forces the conv to *learn* to extract the condition from a spatial channel every layer — wasteful and weak. FiLM injects the condition **multiplicatively** at every block through a dedicated pathway (`emb_proj`), so each block gets a clean, learned, per-channel knob. The same mechanism carries the *class* embedding in the conditional model at the end — conditioning is just "produce a scale/shift (or an additive embedding) from your condition and inject it into the residual stream."


---

# Part 6: Capturing Activations for Skip Connections

A U-Net's decoder needs the encoder's intermediate feature maps (the skip connections). Rather than thread them through return values, Jeremy uses a tiny **monkeypatch** that wraps a module's `forward` so that every call *also* appends its output to a list. This is the `saved()` trick.

---


In [ ]:
#| export
def saved(m, blk):
    m_ = m.forward

    @wraps(m.forward)
    def _f(*args, **kwargs):
        res = m_(*args, **kwargs)
        blk.saved.append(res)
        return res

    m.forward = _f
    return m

**What does the code above do?**

Replaces module `m`'s `forward` with a wrapper that runs the original, stashes the result in `blk.saved`, and returns it — so `m` behaves identically but leaves a copy of its output behind. The deep dive unpacks the mechanics.


## Deep Dive: The `saved()` Forward-Hook Monkeypatch

This is a small but dense piece of Python metaprogramming, in the same spirit as the hooks and callbacks deep dives from earlier notebooks. The goal: **collect every down-path activation the up-path will need, without changing how the blocks are called.**

### Line by line

```python
def saved(m, blk):
    m_ = m.forward                 # 1. remember the ORIGINAL bound forward
    @wraps(m.forward)              # 3. copy name/docstring/signature onto the wrapper
    def _f(*args, **kwargs):
        res = m_(*args, **kwargs)  # 2a. call the original, get its output
        blk.saved.append(res)      # 2b. record the output on the owning block
        return res                 # 2c. return it unchanged to the caller
    m.forward = _f                 # 4. install the wrapper in place of forward
    return m
```

**1 — capture the original first.** `m_ = m.forward` grabs the bound method *before* we overwrite it. If we referenced `m.forward` inside `_f` instead, we would create infinite recursion (the wrapper calling itself). Saving it to a local closure variable is what makes the swap safe.

**2 — the wrapper is transparent.** `_f` runs the real forward, appends the result to `blk.saved` (a list living on the *block that owns* `m`), and returns the result untouched. To any caller, `m(x)` still returns exactly what it always did — the side effect (recording) is invisible.

**3 — `@wraps` preserves identity.** `functools.wraps` copies `__name__`, `__doc__`, `__wrapped__`, etc. from the original onto `_f`, so debugging/printing the module still shows sensible names instead of `_f`.

**4 — install and return.** `m.forward = _f` monkeypatches the instance, and returning `m` lets you wrap inline: `self.down = saved(nn.Conv2d(...), self)` both *builds* the conv and *registers* it to save into `self.saved`.

### Why this design over the obvious alternatives

- **vs. returning activations up the call stack:** the U-Net's down blocks would each have to return `(output, [saved...])` and the parent would splice them together — noisy and error-prone. The monkeypatch keeps `forward` signatures clean.
- **vs. PyTorch forward hooks (`register_forward_hook`):** those work too, but Jeremy's version is a few lines, has zero framework machinery, and puts the saved list exactly where the up-path expects it (on the block).

The one thing to respect: `DownBlock.forward` **resets** `self.saved = []` at the start of every forward pass (seen next), so the list doesn't grow across batches. The wrapper only ever appends; the owner is responsible for clearing.


---

# Part 7: Assembling the U-Net

With the pieces in hand — conditioned residual blocks, optional attention, and the `saved()` hook — we build the encoder (`DownBlock`), decoder (`UpBlock`), and the full `EmbUNetModel`. The trickiest part is the **channel bookkeeping** where up-path blocks concatenate skip connections; the comments below trace it.

---


In [ ]:
#| export
class DownBlock(nn.Module):
    def __init__(self, n_emb, ni, nf, add_down=True, num_layers=1, attn_chans=0):
        super().__init__()
        self.resnets = nn.ModuleList([saved(EmbResBlock(n_emb, ni if i==0 else nf, nf, attn_chans=attn_chans), self)
                                      for i in range(num_layers)])
        self.down = saved(nn.Conv2d(nf, nf, 3, stride=2, padding=1), self) if add_down else nn.Identity()

    def forward(self, x, t):
        self.saved = []
        for resnet in self.resnets: x = resnet(x, t)
        x = self.down(x)
        return x

**What does the code above do?**

One encoder stage. It holds `num_layers` conditioned residual blocks and (unless it is the last stage, `add_down=False`) a stride-2 conv that **halves** the spatial resolution. Crucially, **every** sub-module is wrapped in `saved(..., self)`, so each residual output *and* the downsample output are appended to `self.saved` — these become the skip connections the decoder will consume. `forward` clears `self.saved` at the start (per the deep dive) and runs the blocks in sequence.


In [ ]:
#| export
class UpBlock(nn.Module):
    def __init__(self, n_emb, ni, prev_nf, nf, add_up=True, num_layers=2, attn_chans=0):
        super().__init__()
        self.resnets = nn.ModuleList(
            [EmbResBlock(n_emb, (prev_nf if i==0 else nf)+(ni if (i==num_layers-1) else nf), nf, attn_chans=attn_chans)
            for i in range(num_layers)])
        self.up = upsample(nf) if add_up else nn.Identity()

    def forward(self, x, t, ups):
        for resnet in self.resnets: x = resnet(torch.cat([x, ups.pop()], dim=1), t)
        return self.up(x)

**What does the code above do?**

One decoder stage — the mirror of `DownBlock`, and where the skip connections come back in. In `forward`, each residual block first **concatenates** the running feature map `x` with a saved down-path activation popped off `ups` (`torch.cat([x, ups.pop()], dim=1)`), then processes it; after the blocks, `upsample` doubles the resolution.

The gnarly line is the input-channel arithmetic in `__init__`:

```python
(prev_nf if i==0 else nf) + (ni if (i==num_layers-1) else nf)
```

Read it as **"channels flowing in" + "channels concatenated from the skip"**:

- The first term is the running feature width: `prev_nf` for the first block (coming from the previous, coarser stage), else `nf`.
- The second term is the skip width being `cat`-ed on: `ni` for the *last* block of the stage (which receives the skip from the matching encoder input resolution), else `nf`.

The blocks pop skips in reverse order (`ups.pop()` is LIFO), which is why the last block matches the earliest-saved (widest-context) skip. This bookkeeping is fiddly by nature — the reliable way to trust it is to run a forward pass and confirm shapes line up, which the training cell does implicitly.


In [ ]:
#| export
class EmbUNetModel(nn.Module):
    def __init__( self, in_channels=3, out_channels=3, nfs=(224,448,672,896), num_layers=1, attn_chans=8, attn_start=1):
        super().__init__()
        self.conv_in = nn.Conv2d(in_channels, nfs[0], kernel_size=3, padding=1)
        self.n_temb = nf = nfs[0]
        n_emb = nf*4
        self.emb_mlp = nn.Sequential(lin(self.n_temb, n_emb, norm=nn.BatchNorm1d),
                                     lin(n_emb, n_emb))
        self.downs = nn.ModuleList()
        n = len(nfs)
        for i in range(n):
            ni = nf
            nf = nfs[i]
            self.downs.append(DownBlock(n_emb, ni, nf, add_down=i!=n-1, num_layers=num_layers,
                                        attn_chans=0 if i<attn_start else attn_chans))
        self.mid_block = EmbResBlock(n_emb, nfs[-1])

        rev_nfs = list(reversed(nfs))
        nf = rev_nfs[0]
        self.ups = nn.ModuleList()
        for i in range(n):
            prev_nf = nf
            nf = rev_nfs[i]
            ni = rev_nfs[min(i+1, len(nfs)-1)]
            self.ups.append(UpBlock(n_emb, ni, prev_nf, nf, add_up=i!=n-1, num_layers=num_layers+1,
                                    attn_chans=0 if i>=n-attn_start else attn_chans))
        self.conv_out = pre_conv(nfs[0], out_channels, act=nn.SiLU, norm=nn.BatchNorm2d, bias=False)

    def forward(self, inp):
        x,t = inp
        temb = timestep_embedding(t, self.n_temb)
        emb = self.emb_mlp(temb)
        x = self.conv_in(x)
        saved = [x]
        for block in self.downs: x = block(x, emb)
        saved += [p for o in self.downs for p in o.saved]
        x = self.mid_block(x, emb)
        for block in self.ups: x = block(x, emb, saved)
        return self.conv_out(x)

**What does the code above do?**

The full attention U-Net. Constructor walkthrough:

- `conv_in`: lifts the input (1 channel for Fashion-MNIST) to `nfs[0]` channels.
- `emb_mlp`: the timestep-embedding MLP (`n_temb → n_emb=4·nfs[0] → n_emb`) that post-processes `timestep_embedding` before it feeds every block.
- **downs**: one `DownBlock` per entry of `nfs`, each doubling channels and (except the last) halving resolution. `attn_chans=0 if i<attn_start else attn_chans` means **attention is switched on only from stage `attn_start` onward** — i.e. at the *lower* resolutions where the sequence length `H·W` is small enough for `O(S²)` attention to be cheap.
- `mid_block`: a single conditioned res block at the bottleneck.
- **ups**: the mirror decoder, with attention placed symmetrically (`i >= n - attn_start`). `num_layers+1` blocks per up stage (one extra to consume the extra skip).
- `conv_out`: projects back to `out_channels`.

`forward`: embed `t` → `conv_in` → run downs (collecting skips into `saved`, seeded with the post-`conv_in` map) → `mid_block` → run ups (each popping skips) → `conv_out`. The output is the network's **noise prediction** `ε̂`, same shape as the input image.

The default `nfs=(224,448,672,896)` is the large config; the training cells below use a smaller `(32,64,128,256)` for Fashion-MNIST.


---

# Part 8: Training the Unconditional Model

Standard `miniai` training: Adam, one-cycle LR, mixed precision, MSE loss between the predicted and true noise. Twenty-five epochs.

---


In [ ]:
lr = 1e-2
epochs = 25
opt_func = partial(optim.Adam, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched), MixedPrecision()]
model = EmbUNetModel(in_channels=1, out_channels=1, nfs=(32,64,128,256), num_layers=2)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

**What does the code above do?**

Assembles the training run:

| Component | Choice |
|-----------|--------|
| Model | `EmbUNetModel`, 1→1 channel, `nfs=(32,64,128,256)`, 2 layers/stage, default `attn_chans=8`, `attn_start=1` (attention off at the highest resolution, on below) |
| Loss | `MSELoss` — regress the predicted noise onto the true `ε` |
| Optimizer | Adam, `eps=1e-5` |
| Schedule | OneCycle, `max_lr=1e-2`, over all `epochs·steps` batches |
| Callbacks | device placement, live loss plot, metrics, per-batch LR schedule, mixed precision |

`MixedPrecision()` runs the forward/backward in FP16 for speed and memory. Nothing here is diffusion-specific except the model and the fact that the *targets* are noise — the training loop is the same one from notebook 09 onward.


In [ ]:
learn.fit(epochs)

**What does the code above do?**

Trains for 25 epochs. **What you should see:** a live loss curve descending from ~0.6-ish toward roughly ~0.03 MSE, with train and validation tracking closely (diffusion noise-prediction rarely overfits because the noising is randomized every batch). This takes a while on a GPU; the trained weights are what the sampling section below draws from. (The original notebook's output — a loss plot and a metrics table — is too large to embed here.)


---

# Part 9: Sampling with DDIM

A trained noise-predictor is not yet an image generator — we need a **sampling** loop that starts from pure noise and iteratively denoises. We use **DDIM**, which is faster than ancestral DDPM sampling and has a knob (`eta`) interpolating between deterministic and stochastic sampling. First we set up an evaluator, then the DDIM step, then the loop.

---


In [ ]:
from miniai.fid import ImageEval

**What does the code above do?**

Imports `ImageEval` from notebook 18's FID module — the object that scores generated batches with FID and KID against real data, using a pretrained feature extractor.


In [ ]:
cmodel = torch.load('models/data_aug2.pkl')
del(cmodel[8])
del(cmodel[7])

@inplace
def transformi(b): b[xl] = [F.pad(TF.to_tensor(o), (2,2,2,2))*2-1 for o in b[xl]]

bs = 2048
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=fc.defaults.cpus)

dt = dls.train
xb,yb = next(iter(dt))

ie = ImageEval(cmodel, dls, cbs=[DeviceCB()])

**What does the code above do?**

Sets up the FID/KID evaluator:

- `cmodel = torch.load('models/data_aug2.pkl')` loads a **pretrained Fashion-MNIST classifier** (from the data-augmentation notebook). `del cmodel[8]; del cmodel[7]` chops off its last two layers (the pooling+head), turning the classifier into a **feature extractor** whose penultimate activations are the "Inception-like" features FID is computed on.
- A fresh `transformi` scales pixels to `[-1, 1]` (the range the classifier expects, `*2-1`), and a new `DataLoaders` at batch size 2048 supplies real images.
- `ie = ImageEval(cmodel, dls, ...)` bundles the feature extractor with real-data statistics so `ie.fid(generated)` and `ie.kid(generated)` return distances.


In [ ]:
sz = (2048,1,32,32)

**What does the code above do?**

The shape of a batch to generate: 2048 single-channel 32×32 images. A large batch gives a lower-variance FID/KID estimate.


In [ ]:
#| export
def ddim_step(x_t, noise, abar_t, abar_t1, bbar_t, bbar_t1, eta, sig, clamp=True):
    sig = ((bbar_t1/bbar_t).sqrt() * (1-abar_t/abar_t1).sqrt()) * eta
    x_0_hat = ((x_t-(1-abar_t).sqrt()*noise) / abar_t.sqrt())
    if clamp: x_0_hat = x_0_hat.clamp(-1,1)
    if bbar_t1<=sig**2+0.01: sig=0.  # set to zero if very small or NaN
    x_t = abar_t1.sqrt()*x_0_hat + (bbar_t1-sig**2).sqrt()*noise
    x_t += sig * torch.randn(x_t.shape).to(x_t)
    return x_0_hat,x_t

**What does the code above do?**

One DDIM update — from the current noisy `x_t` to the next, less-noisy `x_{t-1}`. The deep dive derives it; briefly: estimate the clean image `x_0_hat` from the predicted noise, then re-noise it to the next (lower) noise level, with an `eta`-controlled amount of freshly injected randomness.


## Deep Dive: The DDIM Sampling Step

DDPM sampling (notebook 15) walks back through *every* one of ~1000 timesteps, adding fresh noise at each. DDIM (notebook 20) reformulates the reverse process so you can (a) take **far fewer** steps and (b) choose **how stochastic** the process is via a single parameter `eta`. This `ddim_step` implements one such step. Let `ᾱₜ` be the current signal level, `ᾱₜ₋₁` the next (higher) one, and `b̄ = 1−ᾱ` the corresponding noise variances.

### Step 1 — recover the predicted clean image

```python
x_0_hat = (x_t - (1-abar_t).sqrt()*noise) / abar_t.sqrt()
```

This just **inverts the forward equation** `xₜ = √ᾱₜ·x₀ + √(1−ᾱₜ)·ε`. Solving for `x₀` given the model's noise prediction `ε̂ = noise`:

$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\hat\varepsilon}{\sqrt{\bar\alpha_t}}.$$

So at every step the model implicitly produces a **full guess of the final image**, not just a small increment. `clamp(-1, 1)` keeps that guess in valid pixel range (a cheap but effective stabilizer).

### Step 2 — the stochasticity level `sigma`

```python
sig = ((bbar_t1/bbar_t).sqrt() * (1-abar_t/abar_t1).sqrt()) * eta
```

This is the DDIM variance formula. The `eta` factor scales it:

- **`eta = 0`** ⇒ `sig = 0` ⇒ **fully deterministic** DDIM. Given the same starting noise, you always get the same image; sampling is a fixed ODE-like trajectory.
- **`eta = 1`** ⇒ recovers the **stochastic** DDPM-equivalent variance — fresh noise injected each step.
- values in between trade off sample diversity vs. determinism.

The guard `if bbar_t1 <= sig**2 + 0.01: sig = 0.` zeroes `sigma` when it would exceed the available noise budget (which would make the next `sqrt` negative / NaN), especially near the end of sampling.

### Step 3 — re-noise to the next level

```python
x_t = abar_t1.sqrt()*x_0_hat + (bbar_t1 - sig**2).sqrt()*noise
x_t += sig * torch.randn_like(x_t)
```

Rebuild the next-step image from the clean estimate: `√ᾱₜ₋₁·x̂₀` (the signal at the next level) plus `√(b̄ₜ₋₁ − σ²)·ε̂` (deterministic noise pointing along the model's predicted direction) plus `σ·(fresh noise)` (the stochastic part, zero when `eta=0`). Note the split: part of the next noise is *reused* from the model's own prediction (deterministic) and part is *fresh* — the ratio is exactly what `eta` controls.

### The payoff

Return `(x_0_hat, x_t)`: the clean estimate (useful for visualizing the denoising trajectory) and the actual next state. Iterating this a mere ~50–100 times — versus ~1000 for vanilla DDPM — produces high-quality samples, which is why DDIM is the practical default.


In [ ]:
#| export
@torch.no_grad()
def sample(f, model, sz, steps, eta=1., clamp=True):
    model.eval()
    ts = torch.linspace(1-1/steps,0,steps)
    x_t = torch.randn(sz).cuda()
    preds = []
    for i,t in enumerate(progress_bar(ts)):
        t = t[None].cuda()
        abar_t = abar(t)
        noise = model((x_t, t))
        abar_t1 = abar(t-1/steps) if t>=1/steps else torch.tensor(1)
        x_0_hat,x_t = f(x_t, noise, abar_t, abar_t1, 1-abar_t, 1-abar_t1, eta, 1-((i+1)/100), clamp=clamp)
        preds.append(x_0_hat.float().cpu())
    return preds

**What does the code above do?**

The sampling loop. `@torch.no_grad()` and `model.eval()` because we are only doing inference. Steps:

1. `ts = linspace(1-1/steps, 0, steps)` — the descending schedule of timesteps from near-1 (almost pure noise) down to 0.
2. `x_t = randn(sz)` — start from pure Gaussian noise.
3. Each iteration: compute `ᾱₜ`, ask the model for the noise prediction `model((x_t, t))`, compute the next level `ᾱₜ₋₁`, and call the step function `f` (`ddim_step`) to advance `x_t`. The clean estimate `x_0_hat` is stashed each step.
4. Return the list of clean estimates; `preds[-1]` is the final generated batch.

`f` is passed in so the same loop can drive DDIM or other step rules. The `1-((i+1)/100)` argument is the per-step `sig` schedule input the step function uses.


In [ ]:
# set_seed(42)
preds = sample(ddim_step, model, sz, steps=100, eta=1.)
s = (preds[-1]*2)
s.min(),s.max(),s.shape

(tensor(-1.0918), tensor(1.4292), torch.Size([2048, 1, 32, 32]))

**What does the code above do?**

Generates 2048 images in 100 DDIM steps with `eta=1` (stochastic). `s = preds[-1]*2` rescales the final estimate (the model works in the padded `[-0.5,0.5]` convention; `*2` brings it to roughly `[-1,1]` for display/scoring). The printed min/max (`≈ -1.09, 1.43`) show values land near, slightly outside `[-1,1]` — hence the `clamp(-1,1)` when displaying next.


In [ ]:
show_images(s[:25].clamp(-1,1), imsize=1.5)

**What does the code above do?**

Displays the first 25 generated images in a 5×5 grid, clamped to `[-1,1]`. **What you should see:** recognizable but *unlabeled* Fashion-MNIST items — a mix of shirts, shoes, bags, trousers — since this is the *unconditional* model (it generates whatever it likes). (Image output omitted here.)


In [ ]:
ie.fid(s),ie.kid(s),s.shape

(4.058064770194278, 0.010895456187427044, torch.Size([2048, 1, 32, 32]))

**What does the code above do?**

Scores the 2048 samples. **FID ≈ 4.06**, **KID ≈ 0.011** — low is good; these are strong numbers for Fashion-MNIST, indicating the generated distribution closely matches the real one. FID compares Gaussian fits to the Inception-feature means/covariances; KID is a kernel-based, unbiased alternative.


In [ ]:
preds = sample(ddim_step, model, sz, steps=100, eta=1.)
ie.fid(preds[-1]*2)

5.320260029850715

**What does the code above do?**

Re-samples (new random start) at 100 steps and reports FID **≈ 5.32**. The gap from the previous 4.06 is just sampling variance across different random seeds/batches — a reminder that a single FID number has noise and is best read as "around 4–5 here."


In [ ]:
preds = sample(ddim_step, model, sz, steps=50, eta=1.)
ie.fid(preds[-1]*2)

5.243807277315682

In [ ]:
preds = sample(ddim_step, model, sz, steps=50, eta=1.)
ie.fid(preds[-1]*2)

4.963977301033992

**What does the code above do?**

Two runs at **50** steps (half the compute) give FID **≈ 5.24** and **≈ 4.96** — essentially the same quality as 100 steps. This is the whole point of DDIM: you can cut the number of sampling steps substantially with little quality loss, because each step takes a large, well-directed stride rather than a tiny ancestral one.


---

# Part 10: The Class-Conditional Model

The unconditional model generates *some* item; we now want to ask for a *specific* class. The change is small and elegant: learn a **class embedding** and **add** it to the time embedding, so the same FiLM machinery now carries "which class" alongside "how noisy." The data pipeline is tweaked to pass the label through, and a `cond_sample` loop lets us request a category.

---


In [ ]:
def collate_ddpm(b):
    b = default_collate(b)
    (xt,t),eps = noisify(b[xl])
    return (xt,t,b[yl]),eps

**What does the code above do?**

Redefines the collate to also return the **label** `b[yl]`. Now each batch is `((xₜ, t, c), ε)` — the class id `c` rides alongside the noisy image and timestep. Everything else about noising is unchanged.


In [ ]:
@inplace
def transformi(b): b[xl] = [F.pad(TF.to_tensor(o), (2,2,2,2))-0.5 for o in b[xl]]

tds = dsd.with_transform(transformi)
dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))

dl = dls.train
(xt,t,c),eps = b = next(iter(dl))

**What does the code above do?**

Rebuilds the `DataLoaders` (back to the `[-0.5, 0.5]` training convention) with the label-carrying collate. Unpacking a batch now yields `(xt, t, c), eps` — note the extra `c` (the class ids).


In [ ]:
class CondUNetModel(nn.Module):
    def __init__( self, n_classes, in_channels=3, out_channels=3, nfs=(224,448,672,896), num_layers=1):
        super().__init__()
        self.conv_in = nn.Conv2d(in_channels, nfs[0], kernel_size=3, padding=1)
        self.n_temb = nf = nfs[0]
        n_emb = nf*4
        self.cond_emb = nn.Embedding(n_classes, n_emb)
        self.emb_mlp = nn.Sequential(lin(self.n_temb, n_emb, norm=nn.BatchNorm1d),
                                     lin(n_emb, n_emb))
        self.downs = nn.ModuleList()
        for i in range(len(nfs)):
            ni = nf
            nf = nfs[i]
            self.downs.append(DownBlock(n_emb, ni, nf, add_down=i!=len(nfs)-1, num_layers=num_layers))
        self.mid_block = EmbResBlock(n_emb, nfs[-1])

        rev_nfs = list(reversed(nfs))
        nf = rev_nfs[0]
        self.ups = nn.ModuleList()
        for i in range(len(nfs)):
            prev_nf = nf
            nf = rev_nfs[i]
            ni = rev_nfs[min(i+1, len(nfs)-1)]
            self.ups.append(UpBlock(n_emb, ni, prev_nf, nf, add_up=i!=len(nfs)-1, num_layers=num_layers+1))
        self.conv_out = pre_conv(nfs[0], out_channels, act=nn.SiLU, norm=nn.BatchNorm2d, bias=False)

    def forward(self, inp):
        x,t,c = inp
        temb = timestep_embedding(t, self.n_temb)
        cemb = self.cond_emb(c)
        emb = self.emb_mlp(temb) + cemb
        x = self.conv_in(x)
        saved = [x]
        for block in self.downs: x = block(x, emb)
        saved += [p for o in self.downs for p in o.saved]
        x = self.mid_block(x, emb)
        for block in self.ups: x = block(x, emb, saved)
        return self.conv_out(x)

**What does the code above do?**

Nearly identical to `EmbUNetModel`, with **two additions** that turn it conditional:

1. **`self.cond_emb = nn.Embedding(n_classes, n_emb)`** — a lookup table mapping each of the 10 Fashion-MNIST class ids to a learned `n_emb`-dimensional vector.
2. **`emb = self.emb_mlp(temb) + cemb`** — the class embedding is **added to** the processed time embedding. From there it flows through the *same* FiLM pathway into every residual block.

That is the entire conditioning mechanism: because time and class embeddings live in the same space and are simply summed, every block's scale/shift now depends on *both* "how noisy" and "which class." `forward` unpacks `(x, t, c)` instead of `(x, t)` and looks up `cemb = cond_emb(c)`; everything downstream is unchanged. (This version is not `#| export`-ed — it is the notebook's demonstration of conditioning, whereas the exported library keeps the unconditional `EmbUNetModel`.)


In [ ]:
lr = 1e-2
epochs = 25
opt_func = partial(optim.Adam, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched), MixedPrecision()]
model = CondUNetModel(10, in_channels=1, out_channels=1, nfs=(32,64,128,256), num_layers=2)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

**What does the code above do?**

The same training recipe as the unconditional model, now with `CondUNetModel(10, ...)` (10 classes). The loss is still MSE on the noise; the only new signal the model gets is the class id per image, which it must learn to exploit so that its noise prediction becomes class-aware.


In [ ]:
learn.fit(epochs)

**What does the code above do?**

Trains the conditional model for 25 epochs. **What you should see:** a loss curve very similar to the unconditional run — conditioning does not make the regression harder, it just gives the model extra information. (Plot output omitted.)


In [ ]:
sz = (256,1,32,32)

**What does the code above do?**

A smaller generation batch (256) for the conditional demos — we are eyeballing a 5×5 grid per class, not computing a low-variance FID, so fewer images suffice.


In [ ]:
#| export
@torch.no_grad()
def cond_sample(c, f, model, sz, steps, eta=1.):
    ts = torch.linspace(1-1/steps,0,steps)
    x_t = torch.randn(sz).cuda()
    c = x_t.new_full((sz[0],), c, dtype=torch.int32)
    preds = []
    for i,t in enumerate(progress_bar(ts)):
        t = t[None].cuda()
        abar_t = abar(t)
        noise = model((x_t, t, c))
        abar_t1 = abar(t-1/steps) if t>=1/steps else torch.tensor(1)
        x_0_hat,x_t = f(x_t, noise, abar_t, abar_t1, 1-abar_t, 1-abar_t1, eta, 1-((i+1)/100))
        preds.append(x_0_hat.float().cpu())
    return preds

**What does the code above do?**

The conditional sampling loop. It is `sample` with one addition: `c = x_t.new_full((sz[0],), c, ...)` fills a whole batch with the **requested class id** `c`, and the model is called as `model((x_t, t, c))`. So every image in the batch is generated toward the same class. Otherwise the DDIM machinery is identical.


In [ ]:
lbls = dsd['train'].features[yl].names
lbls

['T - shirt / top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

**What does the code above do?**

Fetches the human-readable class names, so we can label generated grids. Index 0 is "T-shirt/top", index 5 "Sandal", etc. — the `cid` we pass to sampling indexes this list.


In [ ]:
set_seed(42)
cid = 0
preds = sample(cid, ddim_step, model, sz, steps=100, eta=1.)
s = (preds[-1]*2)
show_images(s[:25].clamp(-1,1), imsize=1.5, suptitle=lbls[cid])

**What does the code above do?**

Generates class `cid=0` ("T-shirt/top") with `eta=1` (stochastic) and shows a titled 5×5 grid. **What you should see:** 25 varied T-shirt/top images — same category, different instances. (Image omitted.)

> **A code note worth flagging (be honest about it).** This calls `sample(cid, ddim_step, model, ...)`, but `sample` is the *unconditional* loop defined earlier — its first parameter is `f` (the step function), not a class id, and it calls `model((x_t, t))` with no class. The class-conditional loop defined just above is `cond_sample(c, f, model, ...)`. As written, `sample(cid, ...)` would not drive the conditional model correctly (the arguments shift by one and the class is never passed). In the saved notebook this cell nonetheless shows output, which means it was run against a `sample` that behaved conditionally at the time. **To reproduce faithfully, call `cond_sample(cid, ddim_step, model, sz, steps=100, eta=1.)`.** I am flagging this rather than silently "fixing" Jeremy's cell — preserve his code, but know the corrected call.


In [ ]:
set_seed(42)
cid = 0
preds = sample(cid, ddim_step, model, sz, steps=100, eta=0.)
s = (preds[-1]*2)
show_images(s[:25].clamp(-1,1), imsize=1.5, suptitle=lbls[cid])

**What does the code above do?**

The same class-0 generation but with **`eta=0`** — fully deterministic DDIM. **What you should see:** again 25 "T-shirt/top" images, but typically *smoother / less diverse* than the `eta=1` batch, because deterministic sampling injects no fresh noise and the trajectory is fixed by the initial `x_t`. (Same `cond_sample`-vs-`sample` caveat as the previous cell applies.)


---

# Part 11: Export

The final cell compiles all `#| export`-marked cells into `miniai/diffusion.py`.

---


In [ ]:
import nbdev; nbdev.nbdev_export()

**What does the code above do?**

Runs nbdev's exporter, which scans this notebook for `#| export` cells and writes them (in order) to `miniai/diffusion.py` — the schedule (`abar`, `noisify`), `timestep_embedding`, the blocks (`pre_conv`, `upsample`, `lin`, `SelfAttention`, `SelfAttention2D`, `EmbResBlock`, `saved`, `DownBlock`, `UpBlock`, `EmbUNetModel`), and the samplers (`ddim_step`, `sample`, `cond_sample`). After this, other notebooks can simply `from miniai.diffusion import *`.


---

# Summary and What's Next

### What we built

| Piece | Role |
|-------|------|
| `abar` / `noisify` | Cosine forward-noising: input `(xₜ, t)`, target `ε` |
| `timestep_embedding` | Scalar noise level → rich sinusoidal vector |
| FiLM in `EmbResBlock` | `x*(1+scale)+shift` — condition every block on the embedding |
| `SelfAttention2D` | Notebook-27 attention, dropped in at low resolution only |
| `saved()` | Monkeypatch that records down-path activations for skips |
| `EmbUNetModel` | The full attention U-Net noise-predictor |
| `ddim_step` / `sample` | DDIM sampling; `eta` = stochastic↔deterministic |
| `ImageEval` FID/KID | Quality score (~4–5 FID here) |
| `CondUNetModel` / `cond_sample` | Add a class embedding → generate a chosen class |

### The ideas to remember

1. **Conditioning = inject an embedding into the residual stream.** Time enters via a sinusoidal embedding + FiLM scale/shift; class enters by *adding a learned embedding to the time embedding*. Same pathway, richer condition. This generalizes straight to text conditioning (swap the embedding source) and to physical forcings in a climate emulator.
2. **`1 + scale`, not `scale`.** Initializing modulation near identity is why the conditioned network trains stably. A recurring trick: make new machinery start as a no-op.
3. **Attention where the sequence is short.** `attn_start` puts `O(S²)` attention only at downsampled stages — global reasoning where it is cheap. The diffusion analog of letting far-apart regions of a field communicate.
4. **DDIM predicts `x₀` every step, then re-noises.** That "guess the clean image, step back toward it" structure is what lets it take 50 steps instead of 1000, and `eta` dials the stochasticity.
5. **`saved()` and the channel bookkeeping** are fiddly plumbing — trust them by running a forward pass and checking shapes, not by staring.

### Where this goes

This is the architecture that, scaled up and moved into a **latent** space, becomes latent diffusion / Stable Diffusion (notebooks 30–31): the same conditioned attention U-Net, trained to denoise VAE latents instead of pixels, with cross-attention carrying a text condition instead of a class embedding. Notebook 29 (VAE) supplies that latent space.

### Suggested next steps

1. Re-read the four deep dives (timestep embedding, FiLM, `saved()`, DDIM step) — that is where the concepts and any of my errors concentrate. In particular, sanity-check the DDIM variance algebra against your DDIM notebook (20).
2. Note the honest flag on the two conditional-sampling cells: Jeremy's `sample(cid, ...)` calls should be `cond_sample(cid, ...)` to run correctly; the preserved code is verbatim, the corrected call is in the annotation.
3. When satisfied, run `concept-extraction` (candidates: sinusoidal timestep embedding, FiLM scale/shift, `saved()` monkeypatch, attention placement/`attn_start`, DDIM `x₀`-prediction + `eta`, class conditioning by embedding addition).
4. Optionally `/colab` for a GPU-ready version and `/html` to publish.

---
